# Freyra — Influencer Edition (T4-Optimised Colab)

**Run cells top to bottom.**  
Cell 2 installs packages and **automatically restarts the runtime** — that is expected.  
After the restart, skip Cells 1 and 2 and start from **Cell 3**.

> Preset: `influencer` · Model: RealVisXL V5.0 · LoRA: SDXL Film Photography  
> Optimised for Google Colab free-tier T4 GPU (15 GB VRAM)

In [ ]:
# ── Cell 1 · Clone repo ──────────────────────────────────────────────────
import os

REPO_URL  = 'https://github.com/Ashutosh94g/Freyra.git'   # ← your fork
REPO_DIR  = '/content/Freyra'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    print('Repo already cloned — pulling latest changes.')
    os.system(f'git -C {REPO_DIR} pull --ff-only')

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ── Cell 2 · Install dependencies  (runtime auto-restarts after this) ────
#
# Safe to re-run: if numpy is already < 2.0 we skip the heavy install.
#
import sys, subprocess, os
os.chdir('/content/Freyra')

def _run(cmd): subprocess.run(cmd, check=True)

# Cupy causes numpy 2.x crashes — remove it if present
_run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'cupy-cuda12x',
      'cupy-cuda11x', 'cupy'])

# Check if we've already installed (survives kernel restarts via disk cache)
try:
    import numpy as _np
    _ok = tuple(int(x) for x in _np.__version__.split('.')[:2]) < (2, 0)
except Exception:
    _ok = False

if _ok:
    print(f'✅ Already installed (numpy {_np.__version__}). Jump to Cell 3.')
else:
    # PyTorch 2.3 + CUDA 12.1 — matches Colab T4 driver
    print('📦 Installing PyTorch 2.3.0 + CUDA 12.1 …')
    _run([sys.executable, '-m', 'pip', 'install', '-q',
          'torch==2.3.0', 'torchvision==0.18.0',
          '--extra-index-url', 'https://download.pytorch.org/whl/cu121'])

    # Project requirements (numpy<2.0, piexif, etc. — already pinned in file)
    print('📦 Installing project requirements …')
    _run([sys.executable, '-m', 'pip', 'install', '-q',
          '-r', 'requirements_versions.txt'])

    # Downgrade pygit2 to the version Fooocus launch.py expects if needed
    _run([sys.executable, '-m', 'pip', 'install', '-q',
          'pygit2>=1.15.0'])

    print('✅ Done!  Restarting runtime …')
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# ── Cell 3 · Download models (skip if already on disk) ───────────────────
#
# Downloads RealVisXL V5.0 + SDXL Film Photography LoRA from HuggingFace.
# Uses the URLs from presets/influencer.json.
# Safe to skip if you already have the files.
#
import os, urllib.request
os.chdir('/content/Freyra')

MODELS_TO_DOWNLOAD = {
    'models/checkpoints/RealVisXL_V5.0.safetensors':
        'https://huggingface.co/SG161222/RealVisXL_V5.0/resolve/main/RealVisXL_V5.0.safetensors',
    'models/loras/SDXL_FILM_PHOTOGRAPHY_STYLE_V1.safetensors':
        'https://huggingface.co/mashb1t/fav_models/resolve/main/fav/SDXL_FILM_PHOTOGRAPHY_STYLE_V1.safetensors',
}

def _download(dest, url):
    if os.path.exists(dest):
        size_mb = os.path.getsize(dest) / 1e6
        print(f'  ✅ {os.path.basename(dest)} already exists ({size_mb:.0f} MB)')
        return
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    print(f'  ⬇  Downloading {os.path.basename(dest)} …')
    try:
        import subprocess, sys
        subprocess.run(
            ['wget', '-q', '--show-progress', '-O', dest, url],
            check=True
        )
        size_mb = os.path.getsize(dest) / 1e6
        print(f'  ✅ {os.path.basename(dest)} downloaded ({size_mb:.0f} MB)')
    except Exception as e:
        print(f'  ❌ Failed: {e}')

print('Checking / downloading models:')
for dest_path, url in MODELS_TO_DOWNLOAD.items():
    _download(dest_path, url)

print('\nDone. Proceed to Cell 4.')

In [ ]:
# ── Cell 4 · Verify GPU & VRAM before launch ─────────────────────────────
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_gb  = props.total_memory / 1024**3
    free_gb   = (props.total_memory - torch.cuda.memory_reserved(0)) / 1024**3
    print(f'GPU  : {props.name}')
    print(f'VRAM : {total_gb:.1f} GB total  |  {free_gb:.1f} GB free')
    if total_gb < 12:
        print('⚠️  Less than 12 GB VRAM — you may need to lower resolution in the UI.')
    else:
        print('✅ VRAM looks fine for influencer preset.')
else:
    print('❌ No CUDA GPU found. Are you on a GPU runtime? (Runtime → Change runtime type → T4)')

In [ ]:
# ── Cell 5 · Launch Freyra (influencer preset) ────────────────────────────
#
#  --share               public Gradio tunnel URL
#  --always-high-vram    keep all models on GPU (avoids slow CPU<->GPU swap)
#  --unet-in-fp8-e4m3fn  compress UNet VRAM 6.5 GB → 3.3 GB
#  --preset influencer   loads presets/influencer.json
#  --disable-image-log   saves a little RAM on Colab's 12.7 GB system limit
#
import os
os.chdir('/content/Freyra')
os.environ['GRADIO_SERVER_PORT'] = '7865'

os.system(
    'python launch.py '
    '--share '
    '--always-high-vram '
    '--unet-in-fp8-e4m3fn '
    '--preset influencer '
    '--disable-image-log'
)

## Troubleshooting

| Symptom | Fix |
|---|---|
| `RuntimeError: CUDA out of memory` | Lower resolution to `832×1104` in the UI, or enable only 1 LoRA |
| `ModuleNotFoundError: numpy` | Re-run Cell 2 — runtime may have reset |
| Gradio URL not appearing | Wait 60–90 seconds; large model download may still be in progress |
| `pygit2` version error | Cell 2 re-installs it; restart runtime after Cell 2 if needed |
| Black/broken images | Check the LoRA strength — try reducing Film Photography LoRA to 0.15 |

### Resolution guide for T4

| Aspect ratio | Safe resolution | Notes |
|---|---|---|
| Portrait 3:4 | `896×1152` | Default influencer preset |
| Square 1:1 | `1024×1024` | Fine with fp8 UNet |
| Landscape 4:3 | `1152×896` | Fine |
| Tall portrait 2:3 | `832×1248` | Reduce LoRA if OOM |